# Error Case and Shortcut Region Analysis

This notebook reads each model's `test_predictions.csv`, writes high-margin false-positive / false-negative cases, then reloads model checkpoints and image data to summarize Grad-CAM shortcut regions.

Outputs per model:
- `false_positive.csv`
- `false_negative.csv`
- `shortcut_regions.csv`


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
MODELS = ['densenet', 'efficientnet', 'vit']
MAX_CASES = 12
MAX_SHORTCUT_CASES_PER_TYPE = 6

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir: {DATA_DIR}')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')


In [ ]:
from src.analysis.run_error_cases import analyze_model as analyze_error_cases

error_outputs = {}
for model_key in MODELS:
    outputs = analyze_error_cases(CHECKPOINT_DIR, model_key, MAX_CASES)
    error_outputs[model_key] = outputs
    print(
        f"{model_key}: FP={len(outputs['false_positive.csv'])}, "
        f"FN={len(outputs['false_negative.csv'])}"
    )


In [ ]:
import torch
from src.analysis.run_shortcut_regions import _build_image_index, analyze_model as analyze_shortcuts

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
image_index = _build_image_index(DATA_DIR)
print(f'Indexed {len(image_index):,} PNG images')

shortcut_outputs = {}
for model_key in MODELS:
    shortcut_df = analyze_shortcuts(
        model_key=model_key,
        data_dir=DATA_DIR,
        checkpoint_dir=CHECKPOINT_DIR,
        image_index=image_index,
        max_per_type=MAX_SHORTCUT_CASES_PER_TYPE,
        device=device,
    )
    shortcut_outputs[model_key] = shortcut_df
    print(f'\n[{model_key}]')
    print(shortcut_df.to_string(index=False))


In [ ]:
for model_key in MODELS:
    print(f'\n[{model_key}] false_positive.csv')
    display(error_outputs[model_key]['false_positive.csv'])
    print(f'[{model_key}] false_negative.csv')
    display(error_outputs[model_key]['false_negative.csv'])
    print(f'[{model_key}] shortcut_regions.csv')
    display(shortcut_outputs[model_key])
